In [2]:
import sys; sys.path.insert(0, "..")
from src.evaluation import DATASET
from src.retrieval import CHUNKS, label

by_label = {}
for c in CHUNKS["article"]:
    by_label.setdefault(label(c), []).append(c)

for i, row in enumerate(DATASET, 1):
    print(f"\n{'=' * 90}\nQ{i} [{row['cat']}] {row['q']}\nREF: {row['ref']}")
    for lab in row["expected"][:1]:                       # show the first expected article
        print(f"--- {lab} ---\n{by_label[lab][0].page_content[:600]}")

/Users/abdul/Projects/uae-labour-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1023 21:15:49.879000 60941 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10985.34it/s]



Q1 [direct] How many days of annual leave do I get?
REF: 30 days of paid annual leave per year after one year of service; 2 days per month if service is more than 6 months but less than 1 year.
--- labour_law:29 ---
[Labour Law — Article 29, part 1/2]
Article (29)
Annual leave
1. Without prejudice to the Worker's rights acquired for the period preceding the date of
enforcement of the provisions of this Decree by law, the Worker shall be entitled to an
annual leave with full pay of not less than:
a. Thirty days for each year of his extended service.
b. Two days for each month if his service period is more than six months and less than
one year.
c. A leave for parts of the last year he spent at work, in the event that his service ends
before using his annual leave balance.
2. The part-time Worker is entitle

Q2 [multi-doc] Can I carry forward my unused annual leave to next year?
REF: Yes. The worker may carry forward up to half of the annual leave into the following year, or agree with 

In [3]:
from src.evaluation import upload_dataset, DATASET_NAME
ds = upload_dataset()
print("Dataset ready:", DATASET_NAME)

Dataset ready: uae-labour-rag-eval-15


In [4]:
from langsmith import evaluate
from src.evaluation import TARGETS, EVALUATORS

results = {}
for name, target in TARGETS.items():
    print(f"Running {name} ...")
    results[name] = evaluate(
        target, data=DATASET_NAME, evaluators=EVALUATORS,
        experiment_prefix=name, max_concurrency=3,
    )

Running A_naive ...
View the evaluation results for experiment: 'A_naive-f7b38e80' at:
https://smith.langchain.com/o/52839ba0-56ca-4017-964c-d619df92115d/datasets/14c0b8aa-b63e-449c-bf6d-e809c8cd7534/compare?selectedSessions=c8066eba-d6fb-4897-8733-200b8723462f




0it [00:00, ?it/s]

: 

In [ ]:
import pandas as pd
metrics = ["decision_correct", "retrieval_hit", "citation_hit", "correctness", "faithfulness"]
rows = {}
for name, res in results.items():
    df = res.to_pandas()
    rows[name] = {m: df[f"feedback.{m}"].mean() for m in metrics if f"feedback.{m}" in df}
    rows[name]["avg_seconds"] = df["execution_time"].mean()
summary = pd.DataFrame(rows).T.round(2)
summary